# FlashSpec Colab A100 Profiling Runbook

Select `Runtime -> Change runtime type -> GPU` in Colab first. Use A100 for formal profiling.

Recommended order:

1. Mount Drive, confirm GPU, install FlashSpec and Triton.
2. Check `ncu`, then run correctness tests.
3. Run P1 serving allocator validation to check request lifecycle, JSON schema, and allocator stats.
4. Run sanity profiling once and confirm `measured_*` fields are produced.
5. Full matrix profiling is historical; rerun only for reproduction or missing points.
6. Use `doc/optimization-log.md` to choose source-line / instruction attribution cases.
7. Save `.ncu-rep` files and update `optimization-log.md` with conclusions.

Default project directory: `/content/drive/MyDrive/FlashSpecColab`. Outputs are written under `results/colab_kernels/`, `results/colab_serving/`, `results/profile_matrix/`, and `results/ncu_source_attribution/`.


## 0. 挂载 Google Drive 并设置路径

结果会直接写到你的 Drive，避免 Colab runtime 重启后丢失。

In [ ]:
from google.colab import drive
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')
COLAB_RESULTS_DIR = PROJECT_DIR / 'results' / 'colab_kernels'
MATRIX_RESULTS_DIR = PROJECT_DIR / 'results' / 'profile_matrix'

drive.mount('/content/drive')
COLAB_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('COLAB_RESULTS_DIR:', COLAB_RESULTS_DIR)
print('MATRIX_RESULTS_DIR:', MATRIX_RESULTS_DIR)

## 1. 检查 GPU 环境

这里应该看到 `cuda available=True`。如果不是 A100，也可以先跑 correctness 和小规模 sanity，但 profiling 结果不要和 A100 结果直接比较。

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('当前 Colab runtime 没有 CUDA GPU，请切换到 GPU/A100 runtime。')

## 2. 克隆或更新项目

如果 Drive 里已经有仓库，会执行 `git pull --ff-only`。如果你在 notebook 里改了代码，先把改动提交或推送后再运行这一格。

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/honey-floria/FlashSpec.git', str(PROJECT_DIR)],
        check=True,
    )
else:
    raise RuntimeError(
        f'{PROJECT_DIR} 已存在但不是 Git 仓库，请改名或删除后重新 clone FlashSpec。'
    )

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
%pip install -e '.[triton]'

In [ ]:
import os
PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
%pip install -e '.[triton]'

## 3. 检查 FlashSpec、CUDA、Triton 和 ncu

`HAS_TRITON=True` 才能跑 Triton kernel。`ncu` 用来采集 `measured_*` profiler 指标。

In [ ]:
import os
import shutil
import subprocess
import sys
import torch
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from flashspec.triton_kernels import HAS_TRITON

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('HAS_TRITON:', HAS_TRITON)

NCU_BIN = shutil.which('ncu')
if NCU_BIN is None:
    for cand in ['/usr/local/cuda/bin/ncu', '/opt/nvidia/nsight-compute/ncu']:
        if os.path.exists(cand):
            NCU_BIN = cand
            break
if NCU_BIN is None:
    print('未找到 ncu，尝试 apt 安装 nsight-compute ...')
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nsight-compute'], check=False)
    NCU_BIN = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'

if os.path.exists(NCU_BIN):
    os.environ['PATH'] = os.path.dirname(NCU_BIN) + os.pathsep + os.environ.get('PATH', '')
    os.environ['NCU_BIN'] = NCU_BIN
    ver = subprocess.run([NCU_BIN, '--version'], capture_output=True, text=True)
    print('NCU_BIN:', NCU_BIN)
    print(ver.stdout.splitlines()[0] if ver.stdout else ver.stderr[:200])
else:
    os.environ['NCU_BIN'] = 'ncu'
    print('仍然找不到 ncu，后续带 --profile-ncu 的 cell 可能失败。')

## 4. 跑 correctness tests

A100 + Triton + CUDA 环境下先确认 Kernel 1 / Kernel 2 的输出仍然对齐 reference。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab
!python -m unittest discover -s tests

## 5. P1 Serving Allocator Validation

This section does not need `ncu`. First run a small CUDA smoke test to validate allocator lifecycle, JSON schema, and metric fields. Then run an A100-sized serving benchmark and save JSON files under `results/colab_serving/`.

Focus on these fields: `ttft_ms = prefill_ms + first_attention_ms`, `decode_ms`, `tpot_ms`, `tokens_per_second`, `block_utilization`, `fragmentation`, `arrivals`, `finishes`.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
SERVING_OUT = REPO / 'results' / 'colab_serving'
os.chdir(REPO)
SERVING_OUT.mkdir(parents=True, exist_ok=True)

smoke_cmd = [
    sys.executable, 'benchmarks/e2e_serving.py',
    '--requests', '4',
    '--prompt-lens', '64,128',
    '--prompt-length-distribution', 'bimodal',
    '--decode-steps', '4',
    '--request-life-steps', '2',
    '--heads', '4',
    '--head-dim', '64',
    '--block-size', '16',
    '--device', 'cuda',
    '--dtype', 'float16',
    '--json',
]
print('>>', ' '.join(smoke_cmd))
proc = subprocess.run(smoke_cmd, check=True, capture_output=True, text=True)
print(proc.stdout)
smoke = json.loads(proc.stdout)

required_fields = [
    'ttft_ms', 'tpot_ms', 'tokens_per_second', 'prefill_ms', 'decode_ms',
    'block_utilization', 'fragmentation', 'allocated_blocks', 'free_blocks',
    'live_requests', 'used_tokens', 'capacity_tokens', 'padding_waste',
    'arrivals', 'finishes', 'prompt_lens', 'arrival_prefill_ms', 'total_prefill_ms',
]
missing = [key for key in required_fields if key not in smoke]
if missing:
    raise AssertionError(f'missing serving JSON fields: {missing}')
if not str(smoke['device']).startswith('cuda'):
    raise AssertionError(f'expected CUDA serving run, got device={smoke["device"]}')
if smoke['finishes'] <= 0 or smoke['arrivals'] <= smoke['requests']:
    raise AssertionError('request finish/arrival lifecycle was not exercised')
if smoke['block_utilization'] <= 0 or smoke['capacity_tokens'] <= 0:
    raise AssertionError('allocator stats look invalid')

smoke_path = SERVING_OUT / 'serving_p1_smoke_cuda.json'
smoke_path.write_text(json.dumps(smoke, indent=2), encoding='utf-8')
print('smoke json:', smoke_path)
print('schema/lifecycle check: OK')

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
SERVING_OUT = REPO / 'results' / 'colab_serving'
os.chdir(REPO)
SERVING_OUT.mkdir(parents=True, exist_ok=True)

# Default scale validates allocator behavior on A100 without trying to saturate the GPU.
# For a heavier run, increase requests to 32 and decode_steps to 64.
serving_cases = [
    dict(
        name='serving_p1_a100_bimodal_lifecycle',
        requests=16,
        prompt_lens='512,1024,2048',
        prompt_length_distribution='bimodal',
        decode_steps=32,
        request_life_steps=16,
        heads=32,
        head_dim=128,
        block_size=16,
    ),
    dict(
        name='serving_p1_a100_random_steady',
        requests=16,
        prompt_lens='512,1024,2048',
        prompt_length_distribution='random',
        decode_steps=32,
        request_life_steps=0,
        heads=32,
        head_dim=128,
        block_size=16,
    ),
]

summary_rows = []
for case in serving_cases:
    cmd = [
        sys.executable, 'benchmarks/e2e_serving.py',
        '--requests', str(case['requests']),
        '--prompt-lens', case['prompt_lens'],
        '--prompt-length-distribution', case['prompt_length_distribution'],
        '--decode-steps', str(case['decode_steps']),
        '--request-life-steps', str(case['request_life_steps']),
        '--heads', str(case['heads']),
        '--head-dim', str(case['head_dim']),
        '--block-size', str(case['block_size']),
        '--device', 'cuda',
        '--dtype', 'float16',
        '--json',
    ]
    print('\nCASE:', case['name'])
    print('>>', ' '.join(cmd))
    proc = subprocess.run(cmd, check=True, capture_output=True, text=True)
    data = json.loads(proc.stdout)
    out_path = SERVING_OUT / f"{case['name']}.json"
    out_path.write_text(json.dumps(data, indent=2), encoding='utf-8')

    row = {
        'name': case['name'],
        'ttft_ms': data['ttft_ms'],
        'prefill_ms': data['prefill_ms'],
        'arrival_prefill_ms': data['arrival_prefill_ms'],
        'decode_ms': data['decode_ms'],
        'tpot_ms': data['tpot_ms'],
        'tokens_per_second': data['tokens_per_second'],
        'block_utilization': data['block_utilization'],
        'fragmentation': data['fragmentation'],
        'arrivals': data['arrivals'],
        'finishes': data['finishes'],
        'json': str(out_path),
    }
    summary_rows.append(row)
    for key, value in row.items():
        print(f'{key}: {value}')

print('\nserving summary:')
for row in summary_rows:
    print(
        f"{row['name']}: ttft={row['ttft_ms']:.3f} ms, "
        f"tpot={row['tpot_ms']:.3f} ms, tok/s={row['tokens_per_second']:.1f}, "
        f"util={row['block_utilization']:.3f}, frag={row['fragmentation']:.3f}, "
        f"arrivals={row['arrivals']}, finishes={row['finishes']}"
    )


## 6. Sanity Profiling

先只跑两个代表点，确认 `--profile-ncu` 能输出带硬件指标的 JSON。这个 cell 成功后再进入矩阵化 profiling。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
OUT = REPO / 'results' / 'colab_kernels'
os.chdir(REPO)
OUT.mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get('NCU_BIN', 'ncu')

single_points = [
    ('triton_fused', 2048, 128),
    ('triton_paged', 2048, 128),
]

for backend, seq, hd in single_points:
    out = OUT / f'{backend}_s{seq}_d{hd}.json'
    cmd = [
        'python', 'benchmarks/microbench.py', '--backend', backend,
        '--batch', '16', '--heads', '32', '--seq-len', str(seq),
        '--head-dim', str(hd), '--block-size', '16',
        '--iters', '50', '--warmup', '10', '--repeats', '20',
        '--device', 'cuda', '--dtype', 'float16', '--json', '--include-raw',
        '--profile-ncu', '--ncu-bin', NCU_BIN,
        '--output', str(out),
    ]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 7. Matrix Profiling (rerun only for reproduction or missing points)

full matrix + NCU fast metrics 已经说明当前瓶颈：有效访存/调度效率比单纯 occupancy 更重要。

这一格现在默认使用 `dry`，只打印命令不实际运行。只有需要复现矩阵或补缺失点时，再把 `MATRIX_PRESET` 改成 `small` 或 `full`，并打开 `PROFILE_NCU`。


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(REPO)
NCU_BIN = os.environ.get('NCU_BIN', 'ncu')

# 取值：'dry' 只打印命令，'small' 跑关键组合，'full' 跑更完整矩阵。
MATRIX_PRESET = 'dry'
PROFILE_NCU = False

common = [
    '--batch', '16', '--heads', '32', '--block-size', '16',
    '--iters', '50', '--warmup', '10', '--repeats', '20',
    '--device', 'cuda', '--dtype', 'float16',
]
if PROFILE_NCU and MATRIX_PRESET != 'dry':
    common += ['--profile-ncu', '--ncu-bin', NCU_BIN, '--ncu-launch-count', '5', '--ncu-timeout', '900']
if MATRIX_PRESET == 'dry':
    common += ['--dry-run']

if MATRIX_PRESET == 'full':
    fused_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '32,64,128', '--num-warps', '4,8',
        '--num-splits', 'auto,1,4,8', '--length-patterns', 'uniform',
    ]
    paged_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '32,64,128', '--num-warps', '4,8',
        '--length-patterns', 'uniform,descending',
        '--paged-layouts', 'contiguous,shuffled,interleaved',
    ]
else:
    # small：覆盖 block_n=64/128、Split-K 候选，以及 paged locality 的主要对照。
    fused_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '64,128', '--num-warps', '4',
        '--num-splits', 'auto,1,4,8', '--length-patterns', 'uniform',
    ]
    paged_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '64,128', '--num-warps', '4',
        '--length-patterns', 'uniform,descending',
        '--paged-layouts', 'contiguous,shuffled',
    ]

jobs = [
    ['python', 'scripts/profile_matrix.py', '--backend', 'triton_fused',
     '--output-dir', 'results/profile_matrix/fused'] + fused_args + common,
    ['python', 'scripts/profile_matrix.py', '--backend', 'triton_paged',
     '--output-dir', 'results/profile_matrix/paged'] + paged_args + common,
]

for cmd in jobs:
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 8. Summarize JSON, Plots, and Manifests

`colab_kernels` 里的单点结果会生成对比图；`profile_matrix/*_manifest.csv` 会展示矩阵实验清单和每个组合的输出路径。

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
COLAB_RESULTS_DIR = REPO / 'results' / 'colab_kernels'
MATRIX_RESULTS_DIR = REPO / 'results' / 'profile_matrix'
ANALYSIS_DIR = COLAB_RESULTS_DIR / 'analysis'
os.chdir(REPO)

!python scripts/analyze_results.py --results-dir "{COLAB_RESULTS_DIR}" --output-dir "{ANALYSIS_DIR}"

summary = ANALYSIS_DIR / 'summary.csv'
if summary.exists():
    display(pd.read_csv(summary).head(30))

for manifest in sorted(MATRIX_RESULTS_DIR.glob('**/*_manifest.csv')):
    print('\nMANIFEST:', manifest)
    df = pd.read_csv(manifest)
    display(df.head(30))

for png in ['backend_comparison_s2048_d128.png', 'triton_scaling.png']:
    p = ANALYSIS_DIR / png
    if p.exists():
        display(Image(filename=str(p)))

report = REPO / 'results' / 'profile_matrix_report.md'
subprocess.run([
    'python', 'scripts/analyze_matrix.py',
    '--matrix-dir', str(MATRIX_RESULTS_DIR),
    '--source-dir', str(REPO / 'results' / 'ncu_source_attribution_export'),
    '--output', str(report),
], check=True)
print('matrix report:', report)


## 9. Source-Line / Instruction Attribution

根据 `doc/optimization-log.md` 里的 full matrix fast-metrics 归因，下一步不要继续扩大矩阵，而是跑少数能解释差异的归因报告。

要回答的问题：

- 为什么 `block_n=32` occupancy 更高但更慢；
- 为什么 `num_warps=8` 没有收益；
- paged 相对 fused 的 gap 是否来自 block table 间接寻址、地址计算、coalescing/cache 行为或 long scoreboard stall；
- s2048 paged shuffled 慢点具体卡在哪里；
- s4096 best-point 是否能复现 s2048 的归因结论。

下面这一格会把 `.ncu-rep` 导出到 `results/ncu_source_attribution/`。重点查看 `SourceCounters / InstructionStats / MemoryWorkloadAnalysis / SchedulerStats`。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
SOURCE_NCU_DIR = REPO / 'results' / 'ncu_source_attribution'
os.chdir(REPO)
SOURCE_NCU_DIR.mkdir(parents=True, exist_ok=True)

NCU_BIN = os.environ.get('NCU_BIN', 'ncu')
NCU_SECTIONS = ['SourceCounters', 'InstructionStats', 'MemoryWorkloadAnalysis', 'SchedulerStats']
KERNEL_REGEX = {
    'triton_fused': 'fused_dequant_attention|combine_splits',
    'triton_paged': 'paged_quant_attention_kernel|combine_splits',
}

COMMON_BENCH = [
    '--batch', '16', '--heads', '32', '--head-dim', '128', '--block-size', '16',
    '--iters', '50', '--warmup', '10', '--repeats', '1',
    '--device', 'cuda', '--dtype', 'float16', '--seed', '0', '--json',
]

# 来自 optimization-log 的归因点：覆盖 s2048 差异点，并补 s4096 最优点确认。
SOURCE_CASES = [
    dict(name='fused_best_s2048_d128_bn128_nw4_split4', backend='triton_fused', seq_len=2048, block_n=128, num_warps=4, num_splits=4, length_pattern='uniform', paged_layout='contiguous', launch_count=2),
    dict(name='fused_slow_tile_s2048_d128_bn32_nw4_split4', backend='triton_fused', seq_len=2048, block_n=32, num_warps=4, num_splits=4, length_pattern='uniform', paged_layout='contiguous', launch_count=2),
    dict(name='fused_slow_warps_s2048_d128_bn128_nw8_split1', backend='triton_fused', seq_len=2048, block_n=128, num_warps=8, num_splits=1, length_pattern='uniform', paged_layout='contiguous', launch_count=1),
    dict(name='fused_slow_warps_s2048_d128_bn128_nw8_split4', backend='triton_fused', seq_len=2048, block_n=128, num_warps=8, num_splits=4, length_pattern='uniform', paged_layout='contiguous', launch_count=2),
    dict(name='paged_best_s2048_d128_bn128_nw4_uniform_contiguous', backend='triton_paged', seq_len=2048, block_n=128, num_warps=4, num_splits=None, length_pattern='uniform', paged_layout='contiguous', launch_count=1),
    dict(name='paged_locality_slow_s2048_d128_bn128_nw4_uniform_shuffled', backend='triton_paged', seq_len=2048, block_n=128, num_warps=4, num_splits=None, length_pattern='uniform', paged_layout='shuffled', launch_count=1),
    dict(name='fused_best_s4096_d128_bn128_nw4_split4', backend='triton_fused', seq_len=4096, block_n=128, num_warps=4, num_splits=4, length_pattern='uniform', paged_layout='contiguous', launch_count=2),
    dict(name='paged_best_s4096_d128_bn128_nw4_uniform_contiguous', backend='triton_paged', seq_len=4096, block_n=128, num_warps=4, num_splits=None, length_pattern='uniform', paged_layout='contiguous', launch_count=1),
]

RUN_SOURCE_NCU = True  # 改成 False 时只打印命令，不实际运行 ncu。

for case in SOURCE_CASES:
    env = os.environ.copy()
    env['FLASHSPEC_BLOCK_N'] = str(case['block_n'])
    env['FLASHSPEC_NUM_WARPS'] = str(case['num_warps'])
    if case['num_splits'] is None:
        env.pop('FLASHSPEC_NUM_SPLITS', None)
    else:
        env['FLASHSPEC_NUM_SPLITS'] = str(case['num_splits'])

    export_path = SOURCE_NCU_DIR / case['name']
    cmd = [NCU_BIN]
    for section in NCU_SECTIONS:
        cmd += ['--section', section]
    cmd += [
        '--kernel-name', f"regex:{KERNEL_REGEX[case['backend']]}",
        '--launch-count', str(case['launch_count']),
        '--target-processes', 'all',
        '--export', str(export_path),
        '--force-overwrite',
        sys.executable, 'benchmarks/microbench.py',
        '--backend', case['backend'],
        '--seq-len', str(case['seq_len']),
        '--length-pattern', case['length_pattern'],
        '--paged-layout', case['paged_layout'],
        '--layout-seed', '0',
    ] + COMMON_BENCH

    print('\nCASE:', case['name'])
    print('ENV:', ' '.join(f'{k}={env[k]}' for k in ['FLASHSPEC_BLOCK_N', 'FLASHSPEC_NUM_WARPS'] if k in env),
          f"FLASHSPEC_NUM_SPLITS={env.get('FLASHSPEC_NUM_SPLITS', 'unset')}")
    print('>>', ' '.join(cmd))
    if RUN_SOURCE_NCU:
        subprocess.run(cmd, env=env, check=True)

print('\nsource attribution reports:')
for rep in sorted(SOURCE_NCU_DIR.glob('*.ncu-rep')):
    print(rep)
